Mount the drive

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/maryrobynbernabe/coral-sound-project-3-model-training)

In [1]:
TRAIN_CSV = "/kaggle/input/notebooks/maryrobynbernabe/coral-sound-project-2-dataset-splitting/train_set.csv"

Visualize the data

In [2]:
import pandas as pd

# load the train csv
df_train = pd.read_csv(TRAIN_CSV)
df_train.head()

,audio_file_name,RMSE,SPECTRAL_CENTROID,SPECTRAL_BANDWIDTH,ROLLOFF,ZERO_CROSSING_RATE,MFCC_FEATURE_0,MFCC_FEATURE_1,MFCC_FEATURE_2,MFCC_FEATURE_3,...,MFCC_FEATURE_11,MFCC_FEATURE_12,MFCC_FEATURE_13,MFCC_FEATURE_14,MFCC_FEATURE_15,MFCC_FEATURE_16,MFCC_FEATURE_17,MFCC_FEATURE_18,MFCC_FEATURE_19,labels
0,BoN11.0529H.1678278701.180906.4.28.wav,0.013586,3281.713706,3527.383649,7962.826850,0.001039,-495.97717,-25.360006,15.571173,10.742051,...,8.197722,2.783091,7.597062,1.207888,7.518812,4.713681,7.761002,2.533650,7.588442,Healthy
1,BaF5.1330H.671907872.180831.3.18.wav,0.000624,5506.005847,3041.157454,8939.850109,0.254344,-611.53705,-16.766792,34.535435,31.588339,...,2.514375,-3.736288,2.271210,-1.622616,3.087977,-2.768877,3.200042,-0.567399,1.135935,Healthy
2,SaF5.1125R.1678278701.180831.5.33.wav,0.013555,2238.091512,3304.071066,6828.200377,0.000297,-581.68410,-27.747358,29.402313,28.410930,...,6.210212,3.216774,8.026078,1.109000,7.852259,3.715654,7.919775,3.877651,7.079189,Restored
3,SaF2.1112R.671907872.180828.5.49.wav,0.000631,5802.834647,2953.160951,9101.115800,0.419278,-584.80005,-26.345171,26.770414,24.583618,...,6.878403,0.387064,2.824635,-1.029802,4.852212,-1.872255,3.994614,0.126457,2.576796,Restored
4,SaF5.1202D.1677983769.180831.5.30.wav,0.010829,2637.516704,3333.849107,7146.860951,0.000484,-559.42035,-37.722710,3.701616,32.711998,...,6.353807,3.654529,7.881154,1.278509,7.627464,5.473676,8.769265,4.218484,9.438093,Degraded


Convert label column to numbers

In [3]:
LABEL_COLUMN = "labels"

print(df_train[LABEL_COLUMN].value_counts())

labels
Restored    88
Healthy     64
Degraded    57
Name: count, dtype: int64


In [4]:
# convert into numbers
df_train[LABEL_COLUMN] = pd.factorize(df_train[LABEL_COLUMN], sort = True)[0]

Separate features and labels

In [5]:
# gets all features without the labels and discrete features
cols_to_drop = ['audio_file_name', LABEL_COLUMN]
X = df_train.drop(columns=cols_to_drop, axis = 1).values

# all rows
# only the lastb column, which is the label
Y = df_train[LABEL_COLUMN].values

Training the model using cross validation

In [6]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def cross_validation(model, data = (X, Y), splits = 5):
    kf = KFold(n_splits=splits, shuffle=True, random_state=42)

    # Perform k-fold cross-validation
    accuracy = []
    precision = []
    recall = []
    f1 = []

    for train_index, valid_index in kf.split(data[0]):
        X_train, X_valid = data[0][train_index], data[0][valid_index]
        y_train, y_valid = data[1][train_index], data[1][valid_index]

        # Fit the defined model
        model.fit(X_train, y_train)

        # Make predictions on the test data
        y_pred = model.predict(X_valid)

        # Calculate accuracy, precision and recall
        accuracy.append(accuracy_score(y_valid, y_pred))
        precision.append(precision_score(y_valid, y_pred, average = 'weighted'))
        recall.append(recall_score(y_valid, y_pred, average = 'weighted'))
        f1.append(f1_score(y_valid, y_pred, average='weighted'))


    # get arrays
    accuracy_set = np.array(accuracy)
    precision_set = np.array(precision)
    recall_set = np.array(recall)
    f1_set = np.array(f1)

    print("Mean Accuracy: {}".format(accuracy_set.mean()))
    print("Mean Precision: {}".format(precision_set.mean()))
    print("Mean Recall: {}".format(recall_set.mean()))
    print("Mean F1-Score: {}".format(f1_set.mean()))
    return accuracy_set.mean()

KNN

In [7]:
from sklearn.neighbors import KNeighborsClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
knn = KNeighborsClassifier(n_neighbors=3) # vary n_neighbours from 1 to 20

cross_validation(knn)

Mean Accuracy: 0.5689895470383275
Mean Precision: 0.5697572501698472
Mean Recall: 0.5689895470383275
Mean F1-Score: 0.5529958278442841


np.float64(0.5689895470383275)

In [8]:
KNN_Accuracies = []
for i in range (1,20):
  knn = KNeighborsClassifier(n_neighbors=i) # vary n_neighbours from 1 to 20
  KNN_Accuracies.append(cross_validation(knn))


Mean Accuracy: 0.602555168408827
Mean Precision: 0.6131151504322236
Mean Recall: 0.602555168408827
Mean F1-Score: 0.5972500860118946
Mean Accuracy: 0.5644599303135889
Mean Precision: 0.6140614179797413
Mean Recall: 0.5644599303135889
Mean F1-Score: 0.561419035071033
Mean Accuracy: 0.5689895470383275
Mean Precision: 0.5697572501698472
Mean Recall: 0.5689895470383275
Mean F1-Score: 0.5529958278442841
Mean Accuracy: 0.5643437862950058
Mean Precision: 0.5685522551495411
Mean Recall: 0.5643437862950058
Mean F1-Score: 0.5580131299861643
Mean Accuracy: 0.5451800232288038
Mean Precision: 0.5525511875112203
Mean Recall: 0.5451800232288038
Mean F1-Score: 0.5398530951386461
Mean Accuracy: 0.5213704994192799
Mean Precision: 0.524556265996299
Mean Recall: 0.5213704994192799
Mean F1-Score: 0.5144078839193692
Mean Accuracy: 0.48792102206736354
Mean Precision: 0.4859579490067295
Mean Recall: 0.48792102206736354
Mean F1-Score: 0.482504873498371
Mean Accuracy: 0.47351916376306613
Mean Precision: 0.48141

In [9]:
for i in KNN_Accuracies:
  print(i)

0.602555168408827
0.5644599303135889
0.5689895470383275
0.5643437862950058
0.5451800232288038
0.5213704994192799
0.48792102206736354
0.47351916376306613
0.47351916376306624
0.4638792102206736
0.44006968641114985
0.4159117305458769
0.43031358885017423
0.4015098722415796
0.39663182346109177
0.4300813008130081
0.41567944250871075
0.39698025551684085
0.40209059233449473


Random Forest

In [10]:
from sklearn.ensemble import RandomForestClassifier
# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
rf = RandomForestClassifier(n_estimators=50, max_depth = 10) # vary n_estimators from 10 to 100 and max_depth from 1 to 10
cross_validation(rf)

Mean Accuracy: 0.8134727061556329
Mean Precision: 0.8272446726558403
Mean Recall: 0.8134727061556329
Mean F1-Score: 0.8123951365794552


np.float64(0.8134727061556329)

In [11]:
import numpy as np

n_estimators = np.arange(10, 101, 10)
max_depths = np.arange(1, 15, 1)

best_accuracy = []

for i in n_estimators:
  temp = []
  for j in max_depths:
    rf = RandomForestClassifier(n_estimators=i, max_depth = j)
    temp.append(cross_validation(rf))
  best_accuracy.append(temp)

Mean Accuracy: 0.46422764227642277
Mean Precision: 0.4675150268285595
Mean Recall: 0.46422764227642277
Mean F1-Score: 0.3921116556744847


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Mean Accuracy: 0.5644599303135889
Mean Precision: 0.638371134115139
Mean Recall: 0.5644599303135889
Mean F1-Score: 0.5371892224787871
Mean Accuracy: 0.6464576074332171
Mean Precision: 0.6769265158637982
Mean Recall: 0.6464576074332171
Mean F1-Score: 0.6382366847548517
Mean Accuracy: 0.6412311265969803
Mean Precision: 0.6797277528841684
Mean Recall: 0.6412311265969803
Mean F1-Score: 0.6362316432873525
Mean Accuracy: 0.7180023228803716
Mean Precision: 0.7361615467382018
Mean Recall: 0.7180023228803716
Mean F1-Score: 0.7139806149505613
Mean Accuracy: 0.7462253193960511
Mean Precision: 0.7682211465138293
Mean Recall: 0.7462253193960511
Mean F1-Score: 0.7386347346824041
Mean Accuracy: 0.7749128919860627
Mean Precision: 0.8009541917154565
Mean Recall: 0.7749128919860627
Mean F1-Score: 0.7763412810772827
Mean Accuracy: 0.7221835075493612
Mean Precision: 0.7394589556121121
Mean Recall: 0.7221835075493612
Mean F1-Score: 0.7233286971068154
Mean Accuracy: 0.7367015098722416
Mean Precision: 0.7495

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Mean Accuracy: 0.4545876887340302
Mean Precision: 0.5272536797623906
Mean Recall: 0.4545876887340302
Mean F1-Score: 0.373170979427077
Mean Accuracy: 0.6080139372822301
Mean Precision: 0.6899220601311195
Mean Recall: 0.6080139372822301
Mean F1-Score: 0.5824387430163342
Mean Accuracy: 0.6219512195121951
Mean Precision: 0.6649625430864385
Mean Recall: 0.6219512195121951
Mean F1-Score: 0.615977059207778
Mean Accuracy: 0.6987224157955865
Mean Precision: 0.7272235951080065
Mean Recall: 0.6987224157955865
Mean F1-Score: 0.6873893666686822
Mean Accuracy: 0.7657375145180023
Mean Precision: 0.7880325439355571
Mean Recall: 0.7657375145180023
Mean F1-Score: 0.7636151394530792
Mean Accuracy: 0.7463414634146341
Mean Precision: 0.7644701214830143
Mean Recall: 0.7463414634146341
Mean F1-Score: 0.7469235583072004
Mean Accuracy: 0.7895470383275262
Mean Precision: 0.7952130996446639
Mean Recall: 0.7895470383275262
Mean F1-Score: 0.7882675974104301
Mean Accuracy: 0.7704994192799072
Mean Precision: 0.79402

...

Mean Accuracy: 0.478513356562137
Mean Precision: 0.5698936448762232
Mean Recall: 0.478513356562137
Mean F1-Score: 0.38443407409548297
Mean Accuracy: 0.6173054587688734
Mean Precision: 0.7011292267063174
Mean Recall: 0.6173054587688734
Mean F1-Score: 0.6018359245853542
Mean Accuracy: 0.6938443670150987
Mean Precision: 0.743043893388841
Mean Recall: 0.6938443670150987
Mean F1-Score: 0.6872185843822861
Mean Accuracy: 0.689198606271777
Mean Precision: 0.727613713512441
Mean Recall: 0.689198606271777
Mean F1-Score: 0.685922589078638
Mean Accuracy: 0.7656213704994193
Mean Precision: 0.7959490010999882
Mean Recall: 0.7656213704994193
Mean F1-Score: 0.7637111498047809
Mean Accuracy: 0.7845528455284553
Mean Precision: 0.8047532901575739
Mean Recall: 0.7845528455284553
Mean F1-Score: 0.7832576565701703
Mean Accuracy: 0.7797909407665505
Mean Precision: 0.794885980678463
Mean Recall: 0.7797909407665505
Mean F1-Score: 0.7785831847070461
Mean Accuracy: 0.7941927990708478
Mean Precision: 0.8148293680

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Mean Accuracy: 0.4691056910569106
Mean Precision: 0.5365859594518131
Mean Recall: 0.4691056910569106
Mean F1-Score: 0.37983999738594904
Mean Accuracy: 0.5887340301974449
Mean Precision: 0.6726163257457562
Mean Recall: 0.5887340301974449
Mean F1-Score: 0.5682374923807532
Mean Accuracy: 0.6795586527293844
Mean Precision: 0.7204792732238134
Mean Recall: 0.6795586527293844
Mean F1-Score: 0.6741777962843168
Mean Accuracy: 0.7272938443670152
Mean Precision: 0.7669276167974864
Mean Recall: 0.7272938443670152
Mean F1-Score: 0.7247075470516414
Mean Accuracy: 0.751219512195122
Mean Precision: 0.7805164599241255
Mean Recall: 0.751219512195122
Mean F1-Score: 0.7504350026742813
Mean Accuracy: 0.7607433217189314
Mean Precision: 0.7811005804334356
Mean Recall: 0.7607433217189314
Mean F1-Score: 0.7555318428792301
Mean Accuracy: 0.7940766550522648
Mean Precision: 0.8123399495299475
Mean Recall: 0.7940766550522648
Mean F1-Score: 0.7917710825911266
Mean Accuracy: 0.8228803716608596
Mean Precision: 0.8350

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Mean Accuracy: 0.46887340301974445
Mean Precision: 0.5112413583319507
Mean Recall: 0.46887340301974445
Mean F1-Score: 0.38473296320939476
Mean Accuracy: 0.5693379790940767
Mean Precision: 0.6408006051024194
Mean Recall: 0.5693379790940767
Mean F1-Score: 0.548664898891592
Mean Accuracy: 0.6794425087108015
Mean Precision: 0.7264118654158042
Mean Recall: 0.6794425087108015
Mean F1-Score: 0.6723299427082198
Mean Accuracy: 0.7081300813008129
Mean Precision: 0.7501838667634255
Mean Recall: 0.7081300813008129
Mean F1-Score: 0.7071809645871167
Mean Accuracy: 0.7464576074332172
Mean Precision: 0.7799779044029915
Mean Recall: 0.7464576074332172
Mean F1-Score: 0.7438809450239654
Mean Accuracy: 0.7751451800232289
Mean Precision: 0.7934948868436262
Mean Recall: 0.7751451800232289
Mean F1-Score: 0.7733918098996037
Mean Accuracy: 0.7988385598141695
Mean Precision: 0.8144240653190055
Mean Recall: 0.7988385598141695
Mean F1-Score: 0.7976798962707917
Mean Accuracy: 0.7989547038327526
Mean Precision: 0.8

In [12]:
for row in best_accuracy:
  print("\t".join(map(str, row))) # Convert each element to string and join with tabs

0.46422764227642277	0.5644599303135889	0.6464576074332171	0.6412311265969803	0.7180023228803716	0.7462253193960511	0.7749128919860627	0.7221835075493612	0.7367015098722416	0.7796747967479675	0.7652729384436701	0.7659698025551684	0.7607433217189316	0.7655052264808362
0.483855981416957	0.5790940766550523	0.6603948896631823	0.7036004645760745	0.7850174216027874	0.7174216027874565	0.7894308943089431	0.7462253193960511	0.7849012775842044	0.8039488966318235	0.7990708478513356	0.7606271777003485	0.770150987224158	0.7990708478513356
0.4545876887340302	0.6080139372822301	0.6219512195121951	0.6987224157955865	0.7657375145180023	0.7463414634146341	0.7895470383275262	0.7704994192799072	0.7990708478513356	0.8133565621370499	0.7940766550522648	0.7843205574912891	0.8226480836236935	0.7941927990708478
0.49303135888501737	0.5645760743321719	0.6794425087108015	0.6890824622531939	0.7176538908246226	0.78931475029036	0.7847851335656214	0.7797909407665505	0.8087108013937282	0.7794425087108015	0.794076655052

MLP

In [13]:
from sklearn.neural_network import MLPClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
mlp = MLPClassifier(hidden_layer_sizes=(100,), learning_rate_init=0.001, max_iter=20) # vary learning rate as 0.0001, 0.001, 0.01, 0.05, 0.1, 1 and max_iter from 10 to 100

cross_validation(mlp)

Mean Accuracy: 0.35946573751451794
Mean Precision: 0.23764225374801312
Mean Recall: 0.35946573751451794
Mean F1-Score: 0.24784842729891424


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and bein

np.float64(0.35946573751451794)

In [14]:
import numpy as np
from sklearn.neural_network import MLPClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters

learning_rate = [0.01, 0.005, 0.001, 0.0005, 0.0001]
epoch = np.arange(10, 210, 20)

best_accuracy = []

for i in epoch:
  temp = []
  for j in learning_rate:
    mlp = MLPClassifier(hidden_layer_sizes=(100,500), learning_rate_init=0.001, max_iter=20) # vary learning rate as 0.0001, 0.001, 0.01, 0.05, 0.1, 1 and max_iter from 10 to 100
    temp.append(cross_validation(mlp))
  best_accuracy.append(temp)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Ma

Mean Accuracy: 0.3735191637630662
Mean Precision: 0.1750346159605104
Mean Recall: 0.3735191637630662
Mean F1-Score: 0.23434420059953545


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contro

Mean Accuracy: 0.41114982578397213
Mean Precision: 0.22629897785011285
Mean Recall: 0.41114982578397213
Mean F1-Score: 0.2657588317730877


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and bein

...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

Mean Accuracy: 0.3354239256678281
Mean Precision: 0.15018303896607627
Mean Recall: 0.3354239256678281
Mean F1-Score: 0.2024223052905434


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Ma

Mean Accuracy: 0.3443670150987224
Mean Precision: 0.17004033020996162
Mean Recall: 0.3443670150987224
Mean F1-Score: 0.20516816192449044


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [15]:
for row in best_accuracy:
  print("\t".join(map(str, row))) # Convert each element to string and join with tabs

0.3735191637630662	0.41114982578397213	0.3736353077816492	0.36829268292682926	0.4063879210220674
0.3296167247386759	0.335307781649245	0.3354239256678281	0.39198606271777003	0.42113821138211377
0.3735191637630662	0.3449477351916376	0.3491289198606272	0.3448315911730545	0.3159117305458769
0.2962833914053426	0.4020905923344948	0.3590011614401858	0.3301974448315912	0.31126596980255516
0.3016260162601626	0.329616724738676	0.32067363530778165	0.3497096399535424	0.3439024390243902
0.27723577235772356	0.38315911730545876	0.33925667828106854	0.3301974448315912	0.31637630662020905
0.33495934959349594	0.38315911730545876	0.3161440185830429	0.3449477351916376	0.32009291521486644
0.33066202090592334	0.3159117305458769	0.3735191637630662	0.33472706155632986	0.3592334494773519
0.36399535423925666	0.36376306620209065	0.3735191637630662	0.32067363530778165	0.30104529616724734
0.38199767711962834	0.3878048780487805	0.36771196283391405	0.3354239256678281	0.3443670150987224


In [16]:
best_model = RandomForestClassifier(n_estimators=50, max_depth = 10)
best_model.fit(X, Y)

RandomForestClassifier(max_depth=10, n_estimators=50)

In [17]:
import pickle
fh = open("/kaggle/working/rf_best_model.pkl", "wb")
pickle.dump(best_model, fh)
fh.close()